# AI624 PA3 – Quantization
**Due:** 23rd October 2026

In [1]:
%cd AIPA3/

/kaggle/working/AIPA3


In [1]:
!ls

AIPA3


In [13]:
!git pull  https://github.com/Mohammad-Shayan-Asim/AIPA3

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 873 bytes | 291.00 KiB/s, done.
From https://github.com/Mohammad-Shayan-Asim/AIPA3
 * branch            HEAD       -> FETCH_HEAD
Updating 13a6a18..780468e
Fast-forward
 gpt2p.py       |   2 +-
 notebook.ipynb | 375 +++++----------------------------------------------------
 2 files changed, 28 insertions(+), 349 deletions(-)


In [2]:
!git clone https://github.com/Mohammad-Shayan-Asim/AIPA3

Cloning into 'AIPA3'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 13 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 276.31 KiB | 9.21 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [5]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [15]:
!pip install imp

ERROR: Could not find a version that satisfies the requirement imp (from versions: none)
ERROR: No matching distribution found for imp


In [23]:
import importlib
# importlib.reload(gpt2p_data)
importlib.reload(gpt2p)


<module 'gpt2p' from '/kaggle/working/AIPA3/gpt2p.py'>

In [24]:
import torchaudio  # load before transformers to avoid circular import
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel
import os, tempfile, tqdm

from gpt2_data import WikiText2Data
import gpt2p

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : Tesla T4
Memory : 15.6 GB


---
## Task 1 – GPT-2 Baseline and Model Profiling

### 1.1 Model Architecture and Quantization Scope

**GPT-2 small** is a decoder-only autoregressive Transformer with:
- Vocabulary size $V = 50{,}257$, hidden dimension $d = 768$, 12 attention heads, MLP intermediate dimension $4d = 3072$, 12 transformer blocks.
- A learned token embedding matrix $W_E \in \mathbb{R}^{V \times d}$ and a learned positional embedding $W_P \in \mathbb{R}^{1024 \times d}$.
- A final LayerNorm followed by a vocabulary projection **tied** to $W_E$ (i.e. `lm_head.weight` $\equiv$ `transformer.wte.weight`).

**Order of operations within one transformer block** (pre-LayerNorm formulation):

$$
\begin{aligned}
h' &= x + \text{Attn}(\text{LN}_1(x)) \\
h  &= h' + \text{MLP}(\text{LN}_2(h'))
\end{aligned}
$$

where the attention sub-layer expands to:

$$
\text{Attn}(u) = \underbrace{\text{c\_proj}}_{\text{out proj}}\!\left(\,\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)\!V\right), \quad [Q,K,V] = \underbrace{\text{c\_attn}}_{\text{QKV proj}}(u)
$$

and the MLP sub-layer to:

$$
\text{MLP}(u) = \underbrace{\text{c\_proj}}_{\text{out proj}}\!\left(\text{GELU}\!\left(\underbrace{\text{c\_fc}}_{\text{in proj}}(u)\right)\right)
$$

**Quantization scope (Task 2 – all 12 blocks):**

| Projection | GPT-2 module |
|---|---|
| Query / Key / Value | `transformer.h[i].attn.c_attn` |
| Attention output | `transformer.h[i].attn.c_proj` |
| MLP input | `transformer.h[i].mlp.c_fc` |
| MLP output | `transformer.h[i].mlp.c_proj` |

Weights **and** input activations of these projections are quantized. Biases, embeddings, the tied vocab head, LayerNorm, GELU, softmax, residual additions, and attention-score / AV multiplications remain in FP32. These modules use `Conv1D` (weight shape `[in, out]`), not `nn.Linear` (`[out, in]`).

In [7]:
model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")
model.eval()
print(model.config)
print("\n--- Block 0 ---")
print(model.transformer.h[0])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.20.2",
  "use_cache": true,
  "vocab_size": 50257
}


--- Block 0 ---
GPT2Block(

In [8]:
# Verify Conv1D weight layout for the projection modules
c_attn = model.transformer.h[0].attn.c_attn
c_proj = model.transformer.h[0].attn.c_proj
c_fc   = model.transformer.h[0].mlp.c_fc
print(f"c_attn weight shape : {c_attn.weight.shape}  (in=768, out=2304 = 3x768)")
print(f"c_proj weight shape : {c_proj.weight.shape}  (in=768, out=768)")
print(f"c_fc   weight shape : {c_fc.weight.shape}    (in=768, out=3072)")

c_attn weight shape : torch.Size([768, 2304])  (in=768, out=2304 = 3x768)
c_proj weight shape : torch.Size([768, 768])  (in=768, out=768)
c_fc   weight shape : torch.Size([768, 3072])    (in=768, out=3072)


### 1.2 Baseline Language-Model Performance

In [9]:
# Build dataset (seed=42, calib_size=128, seq_len=128)
data        = WikiText2Data(batch_size=8, num_workers=2)
test_loader = data.test_dataloader()

train: 18521 blocks, 117 tokens discarded
validation: 1914 blocks, 112 tokens discarded
test: 2193 blocks, 40 tokens discarded


In [10]:
model.to(device)

total_nll     = 0.0
total_correct = 0
total_targets = 0

with torch.no_grad():
    for (input_ids,) in tqdm.tqdm(test_loader, desc="Evaluating"):
        input_ids = input_ids.to(device)
        logits    = model(input_ids, use_cache=False).logits  # (B, 128, V)

        # Scored positions: 1..127  ->  logits[:, :-1] predicts input_ids[:, 1:]
        shift_logits = logits[:, :-1].contiguous()            # (B, 127, V)
        shift_labels = input_ids[:, 1:].contiguous()          # (B, 127)

        total_nll     += F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            reduction="sum",
        ).item()
        total_correct += (shift_logits.argmax(-1) == shift_labels).sum().item()
        total_targets += shift_labels.numel()

nll = total_nll / total_targets
ppl = torch.exp(torch.tensor(nll)).item()
acc = 100.0 * total_correct / total_targets

print(f"Scored targets : {total_targets:,}")
print(f"Top-1 Accuracy : {acc:.2f}%")
print(f"NLL            : {nll:.4f}")
print(f"Perplexity     : {ppl:.2f}")

Evaluating: 100%|██████████| 275/275 [00:22<00:00, 12.35it/s]

Scored targets : 278,511
Top-1 Accuracy : 29.84%
NLL            : 4.1623
Perplexity     : 64.22


### 1.3 Baseline Profiling

In [11]:
# --- Model storage ---
# model.parameters() deduplicates tied weights (lm_head.weight = wte.weight)
n_unique_params = sum(p.numel() for p in model.parameters())

with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
    tmp_path = f.name
torch.save(model.state_dict(), tmp_path)
size_mb = os.path.getsize(tmp_path) / 1e6
os.remove(tmp_path)

print(f"Unique parameters : {n_unique_params:,}")
print(f"Serialized size   : {size_mb:.1f} MB")

Unique parameters : 124,439,808
Serialized size   : 497.8 MB


In [27]:
# --- Runtime / memory / power profile ---
# Profiling uses batch_size=1, seq_len=128, 10 warm-up + 30 measured passes
from transformers import GPT2Tokenizer
result = gpt2p.profile(model, input_shape=(1, 128), device=device, tokenizer=GPT2Tokenizer.from_pretrained("gpt2"))

lat_ms    = result["total_latency"] * 1e3
seq_per_s = 1.0   / result["total_latency"]
tok_per_s = 128.0 / result["total_latency"]
# torch.cuda.power_draw() returns mW; energy_j is therefore in mJ
power_mw  = result["avg_power_w"]
energy_mj = result["energy_j"]

rows = [
    ("Parameters (calflops)",  result["parameters"]),
    ("MACs",                   result["macs"]),
    ("FLOPs",                  result["flops"]),
    ("Latency (ms)",           f"{lat_ms:.2f}"),
    ("Sequences / s",          f"{seq_per_s:.2f}"),
    ("Tokens / s",             f"{tok_per_s:.1f}"),
    ("GPU mem peak (MB)",      f"{result['gpu_mem_peak'] / 1e6:.1f}"),
    ("GPU mem avg (MB)",       f"{result['gpu_mem_avg']  / 1e6:.1f}"),
    ("Avg power (mW)",         f"{power_mw:.1f}"),
    ("Energy / fwd pass (mJ)", f"{energy_mj:.3f}"),
]

print(f"{'Metric':<28} Value")
print("-" * 50)
for name, val in rows:
    print(f"{name:<28} {val}")

AttributeError: GPT2Tokenizer has no attribute encode_plus

---
## Task 2 – GPT-2 Post-Training Quantization

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
from quant import wrap_gpt2, set_mode, TARGETS

In [ ]:
# ── Shared helpers ──────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate(model, loader, device):
    """Returns (top1_acc%, nll, ppl)."""
    model.eval()
    total_nll, n_correct, n_total = 0.0, 0, 0
    for (ids,) in loader:
        ids          = ids.to(device)
        logits       = model(ids, use_cache=False).logits
        s_logits     = logits[:, :-1].contiguous()
        s_labels     = ids[:, 1:].contiguous()
        total_nll   += F.cross_entropy(
            s_logits.view(-1, s_logits.size(-1)), s_labels.view(-1), reduction="sum"
        ).item()
        n_correct   += (s_logits.argmax(-1) == s_labels).sum().item()
        n_total     += s_labels.numel()
    nll = total_nll / n_total
    return 100.0 * n_correct / n_total, nll, torch.exp(torch.tensor(nll)).item()


@torch.no_grad()
def calibrate(model, loader, device):
    """One forward pass through calibration data (model must be in 'observe' mode)."""
    model.eval()
    for (ids,) in loader:
        model(ids.to(device), use_cache=False)


def weight_recon_errors(fp32_model, quant_model, eps=1e-8):
    """Normalised Frobenius weight reconstruction error per projection."""
    errors = {}
    for i in range(12):
        for parent, attr in TARGETS:
            w_fp  = getattr(getattr(fp32_model.transformer.h[i], parent), attr).weight.detach().float()
            w_q   = getattr(getattr(quant_model.transformer.h[i], parent), attr).weight.detach().float()
            errors[f"h{i}.{parent}.{attr}"] = ((w_fp - w_q).norm()**2 / (w_fp.norm()**2 + eps)).item()
    return errors


@torch.no_grad()
def activation_recon_errors(quant_model, loader, device, n_batches=8, eps=1e-8):
    """Normalised activation reconstruction error per projection via forward pre-hooks.
    Model must be in 'static' or 'dynamic' mode before calling."""
    sums, counts, handles = {}, {}, []
    for i, block in enumerate(quant_model.transformer.h):
        for parent, attr in TARGETS:
            wrapper = getattr(getattr(block, parent), attr)
            key     = f"h{i}.{parent}.{attr}"
            sums[key], counts[key] = 0.0, 0

            def make_hook(w, k):
                def hook(mod, inp):
                    x = inp[0].detach().float()
                    if w.smooth is not None:
                        x = x / w.smooth.float()
                    x_q = w.aq.quantize(x.clone())
                    sums[k]   += ((x - x_q).norm()**2 / (x.norm()**2 + eps)).item()
                    counts[k] += 1
                return hook

            handles.append(wrapper.register_forward_pre_hook(make_hook(wrapper, key)))

    quant_model.to(device)
    for b, (ids,) in enumerate(loader):
        if b >= n_batches:
            break
        quant_model(ids.to(device), use_cache=False)

    for h in handles:
        h.remove()
    return {k: sums[k] / counts[k] if counts[k] else 0.0 for k in sums}


def serialized_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        p = f.name
    torch.save(model.state_dict(), p)
    mb = os.path.getsize(p) / 1e6
    os.remove(p)
    return mb


print("Helpers defined.")

### 2.1 Static and Dynamic Linear PTQ

#### Quantized projection expression

For a linear projection $Y = XW + b$ with row-vector convention:

**Weight** — symmetric per-output-channel, $b$-bit, zero-point $Z_W = 0$:
$$S_W^{(c)} = \frac{\max_i |W_{ic}|}{Q_b}, \qquad \hat{W}_{ic} = S_W^{(c)}\,Q_W^{(c)}, \qquad Q_W^{(c)} = \operatorname{clip}\!\left(\operatorname{round}\!\left(\frac{W_{ic}}{S_W^{(c)}}\right),\,-Q_b,\,Q_b\right)$$

**Activation** — asymmetric per-tensor, unsigned 8-bit:
$$S_X = \frac{x_{\max}-x_{\min}}{255}, \qquad Z_X = \operatorname{clip}\!\left(\operatorname{round}\!\left(\frac{-x_{\min}}{S_X}\right),0,255\right), \qquad \hat{X}_{tj} = S_X(Q_X - Z_X)$$

**Quantized projection** per output channel $c$:
$$Y_{tc} = S_X S_W^{(c)}\underbrace{\left(\sum_j Q_{X,tj}\,Q_{W,jc}^{(c)} - Z_X\sum_j Q_{W,jc}^{(c)}\right)}_{\text{INT accumulation + zero-point correction}} + b_c$$

The zero-point correction $Z_X \sum_j Q_{W,jc}$ is constant per output channel and can be precomputed. Rescaling by $S_X S_W^{(c)}$ and bias addition are done in FP32.

> **Implementation note:** `QuantWrapper` stores dequantized FP32 weights and runs FP32 `addmm` (simulated quantization). The actual serialized size is therefore FP32 + scale buffers; theoretical packed sizes are reported separately below.

In [ ]:
baseline    = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").eval()
calib_loader = data.calib_dataloader()
val_loader   = data.val_dataloader()

# theoretical weight counts for packed-size estimation
proj_params = sum(
    getattr(getattr(baseline.transformer.h[0], p), a).weight.numel()
    for p, a in TARGETS
) * 12  # 12 blocks
total_params = sum(p.numel() for p in baseline.parameters())
other_params = total_params - proj_params
print(f"Projection params : {proj_params:,}")
print(f"Other params      : {other_params:,}  (kept FP32)")

In [ ]:
CONFIGS = [
    ("Static W8A8",  8, "static"),
    ("Dynamic W8A8", 8, "dynamic"),
    ("Static W4A8",  4, "static"),
    ("Dynamic W4A8", 4, "dynamic"),
]

results_21 = {}

for label, w_bw, mode in CONFIGS:
    m = copy.deepcopy(baseline)
    if mode == "static":
        wrap_gpt2(m, w_bw=w_bw, a_bw=8, mode="observe")
        m.to(device)
        calibrate(m, calib_loader, device)
        set_mode(m, "static")
    else:
        wrap_gpt2(m, w_bw=w_bw, a_bw=8, mode="dynamic")
        m.to(device)

    acc, nll, ppl = evaluate(m, test_loader, device)
    w_err         = weight_recon_errors(baseline, m)
    a_err         = activation_recon_errors(m, val_loader, device)
    size_mb       = serialized_size_mb(m)

    # theoretical packed size: INT weights + FP32 scales + FP32 rest
    bytes_per_w  = w_bw / 8
    # w_scale: 1 FP32 per output channel per projection; n_out = proj_params / n_in
    n_scales     = sum(
        getattr(getattr(m.transformer.h[0], p), a).w_scale.numel()
        for p, a in TARGETS
    ) * 12
    theory_mb    = (proj_params * bytes_per_w + n_scales * 4 + other_params * 4) / 1e6

    results_21[label] = dict(
        acc=acc, nll=nll, ppl=ppl,
        size_mb=size_mb, theory_mb=theory_mb,
        w_err_mean=np.mean(list(w_err.values())),
        a_err_mean=np.mean(list(a_err.values())),
        w_err=w_err, a_err=a_err,
    )
    print(f"{label:15s}  Acc={acc:.2f}%  NLL={nll:.4f}  PPL={ppl:.2f}  "
          f"size={size_mb:.1f}MB (theory {theory_mb:.1f}MB)  "
          f"w_err={results_21[label]['w_err_mean']:.4f}  a_err={results_21[label]['a_err_mean']:.4f}")

In [ ]:
# Profile each config (batch_size=1, seq_len=128)
# Rebuild models in static mode for profiling (deepcopy already consumed above)
profile_21 = {}

for label, w_bw, mode in CONFIGS:
    m = copy.deepcopy(baseline)
    if mode == "static":
        wrap_gpt2(m, w_bw=w_bw, a_bw=8, mode="observe")
        m.to(device)
        calibrate(m, calib_loader, device)
        set_mode(m, "static")
    else:
        wrap_gpt2(m, w_bw=w_bw, a_bw=8, mode="dynamic")
        m.to(device)

    r = gpt2p.profile(m, input_shape=(1, 128), device=device)
    profile_21[label] = r
    print(f"{label:15s}  lat={r['total_latency']*1e3:.2f}ms  "
          f"mem_peak={r['gpu_mem_peak']/1e6:.1f}MB  "
          f"power={r['avg_power_w']:.1f}mW  energy={r['energy_j']:.3f}mJ")
    del m

In [ ]:
# ── Summary table 2.1 ───────────────────────────────────────────────────────
hdr  = f"{'Config':<16} {'Acc%':>6} {'NLL':>7} {'PPL':>7} {'Size MB':>8} {'Theory MB':>10} {'W-err':>8} {'A-err':>8} {'Lat ms':>8} {'Tok/s':>8}"
sep  = "-" * len(hdr)
print(hdr)
print(sep)

for label, _, _ in CONFIGS:
    r  = results_21[label]
    pr = profile_21[label]
    print(
        f"{label:<16} {r['acc']:>6.2f} {r['nll']:>7.4f} {r['ppl']:>7.2f} "
        f"{r['size_mb']:>8.1f} {r['theory_mb']:>10.1f} "
        f"{r['w_err_mean']:>8.4f} {r['a_err_mean']:>8.4f} "
        f"{pr['total_latency']*1e3:>8.2f} {128/pr['total_latency']:>8.1f}"
    )

In [ ]:
# ── Layer-wise reconstruction errors ────────────────────────────────────────
# Compare Static vs Dynamic at W8 and W4
pairs = [("Static W8A8", "Dynamic W8A8"), ("Static W4A8", "Dynamic W4A8")]
keys  = [f"h{i}.{p}.{a}" for i in range(12) for p, a in TARGETS]

for s_label, d_label in pairs:
    print(f"\n{'Layer':<28}  {'W-err (S)':>10}  {'W-err (D)':>10}  {'A-err (S)':>10}  {'A-err (D)':>10}")
    print("-" * 75)
    for k in keys[:8]:  # first 2 blocks for brevity; full data in report
        print(
            f"{k:<28}  "
            f"{results_21[s_label]['w_err'][k]:>10.4f}  "
            f"{results_21[d_label]['w_err'][k]:>10.4f}  "
            f"{results_21[s_label]['a_err'][k]:>10.4f}  "
            f"{results_21[d_label]['a_err'][k]:>10.4f}"
        )

### 2.2 SmoothQuant and Outlier Analysis

SmoothQuant (Xiao et al., 2022) reduces activation outliers by absorbing part of the quantization difficulty into the weights via an equivalent channel-wise scaling transformation.

For $Y = XW + b$, introduce $D = \operatorname{diag}(s)$ so that:
$$X' = XD^{-1}, \quad W' = DW, \quad X'W' = XW$$

The per-channel scale is:
$$s_j = \frac{(\max_i |X_{ij}| + \epsilon)^{\alpha}}{(\max_i |W_{ji}| + \epsilon)^{1-\alpha}}$$

where $j$ indexes input channels, the activation maximum is estimated from calibration data, and $\alpha \in [0,1]$ controls the balance between activation and weight difficulty.

In [ ]:
# ── Collect per-channel statistics from the original model ──────────────────
# For blocks 0, 5, 11: record all activation inputs and weight tensors

INSPECT_BLOCKS  = [0, 5, 11]
INSPECT_PROJS   = [("attn", "c_attn"), ("mlp", "c_fc")]   # representative projections

orig = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").eval().to(device)

# act_data[key] = list of tensors with shape (n_tokens, in_features)
act_data = {}
handles  = []

for i in INSPECT_BLOCKS:
    for parent, attr in INSPECT_PROJS:
        layer = getattr(getattr(orig.transformer.h[i], parent), attr)
        key   = f"h{i}.{parent}.{attr}"
        act_data[key] = []

        def make_hook(k):
            def hook(mod, inp):
                x = inp[0].detach().cpu().float()  # (B, T, in)
                act_data[k].append(x.view(-1, x.size(-1)))
            return hook

        handles.append(layer.register_forward_pre_hook(make_hook(key)))

with torch.no_grad():
    for (ids,) in calib_loader:
        orig(ids.to(device), use_cache=False)

for h in handles:
    h.remove()

# Concatenate: (n_total_tokens, in_features)
act_data = {k: torch.cat(v, dim=0) for k, v in act_data.items()}
print({k: v.shape for k, v in act_data.items()})

In [ ]:
# ── Distribution plots (before smoothing) ───────────────────────────────────
# Requires matplotlib

fig, axes = plt.subplots(len(INSPECT_BLOCKS), len(INSPECT_PROJS) * 2,
                         figsize=(16, 4 * len(INSPECT_BLOCKS)))

for row, i in enumerate(INSPECT_BLOCKS):
    for col, (parent, attr) in enumerate(INSPECT_PROJS):
        key = f"h{i}.{parent}.{attr}"
        layer = getattr(getattr(orig.transformer.h[i], parent), attr)

        acts = act_data[key].numpy()          # (n_tokens, in_features)
        wts  = layer.weight.detach().cpu().float().numpy()  # (in, out)

        ch_max_act = np.abs(acts).max(axis=0)   # (in_features,)
        ch_max_wt  = np.abs(wts).max(axis=1)    # (in_features,)

        # Histogram of activation values
        ax = axes[row, col * 2]
        ax.hist(acts.ravel(), bins=100, color="steelblue", alpha=0.8)
        ax.set_title(f"Block {i} {parent}.{attr} – activation dist")
        ax.set_xlabel("Value")
        ax.set_ylabel("Count")

        # Per-channel max absolute magnitude
        ax2 = axes[row, col * 2 + 1]
        ax2.plot(np.sort(ch_max_act)[::-1], label="Activation", color="steelblue")
        ax2.plot(np.sort(ch_max_wt)[::-1],  label="Weight row",  color="darkorange")
        ax2.set_title(f"Block {i} {parent}.{attr} – per-channel max |·|")
        ax2.set_xlabel("Channel rank")
        ax2.set_ylabel("|x|")
        ax2.legend()

        # Report max and 99th percentile
        print(f"{key}  act max={ch_max_act.max():.3f}  act p99={np.percentile(ch_max_act, 99):.3f}  "
              f"wt max={ch_max_wt.max():.3f}  wt p99={np.percentile(ch_max_wt, 99):.3f}")

plt.tight_layout()
plt.savefig("dist_before_smooth.png", dpi=120)
plt.show()

In [ ]:
# ── Smooth scale computation ─────────────────────────────────────────────────

@torch.no_grad()
def compute_smooth_scales(model, calib_loader, device, alpha=0.5):
    """Returns a dict {key: scale_tensor} for all 48 projections."""
    # Step 1: collect per-input-channel activation maxima on calibration data
    act_max  = {}
    handles  = []

    for i, block in enumerate(model.transformer.h):
        for parent, attr in TARGETS:
            layer = getattr(getattr(block, parent), attr)
            key   = f"h.{i}.{parent}.{attr}"
            act_max[key] = None

            def make_hook(k):
                def hook(mod, inp):
                    x     = inp[0].detach().float()              # (B, T, in)
                    x_max = x.abs().amax(dim=(0, 1))             # (in_features,)
                    act_max[k] = x_max if act_max[k] is None else torch.maximum(act_max[k], x_max)
                return hook
            handles.append(layer.register_forward_pre_hook(make_hook(key)))

    model.eval()
    for (ids,) in calib_loader:
        model(ids.to(device), use_cache=False)
    for h in handles:
        h.remove()

    # Step 2: compute s_j = (act_max_j + eps)^alpha / (wt_max_j + eps)^(1-alpha)
    scales = {}
    eps    = 1e-8
    for i, block in enumerate(model.transformer.h):
        for parent, attr in TARGETS:
            layer  = getattr(getattr(block, parent), attr)
            key    = f"h.{i}.{parent}.{attr}"
            w      = layer.weight.detach().float()    # (in, out)
            w_max  = w.abs().amax(dim=1)              # (in,) max over output channels
            x_max  = act_max[key].float()
            scales[key] = (x_max + eps).pow(alpha) / (w_max + eps).pow(1 - alpha)

    return scales


print("compute_smooth_scales defined.")

In [ ]:
# ── Verify floating-point equivalence (alpha=0.5) ────────────────────────────
scales_test = compute_smooth_scales(orig, calib_loader, device, alpha=0.5)

m_smooth_test = copy.deepcopy(orig)
wrap_gpt2(m_smooth_test, w_bw=None, a_bw=8, mode="off", smooth=scales_test)  # w_bw=None keeps weights FP32
m_smooth_test.to(device)

# Compare outputs on a fixed batch
with torch.no_grad():
    ids_sample = next(iter(calib_loader))[0][:2].to(device)
    out_orig   = orig(ids_sample, use_cache=False).logits
    out_smooth = m_smooth_test(ids_sample, use_cache=False).logits
    abs_diff   = (out_orig - out_smooth).abs()

print(f"Max absolute output diff  : {abs_diff.max().item():.6f}")
print(f"Mean relative output diff : {(abs_diff / (out_orig.abs() + 1e-8)).mean().item():.6f}")

In [ ]:
# ── Alpha search on validation set ──────────────────────────────────────────
# Evaluate W8A8 static perplexity for alpha in {0.25, 0.5, 0.75}

ALPHAS = [0.25, 0.5, 0.75]
alpha_results = {}

for alpha in ALPHAS:
    scales = compute_smooth_scales(orig, calib_loader, device, alpha=alpha)

    m = copy.deepcopy(orig)
    wrap_gpt2(m, w_bw=8, a_bw=8, mode="observe", smooth=scales)
    m.to(device)
    calibrate(m, calib_loader, device)   # recalibrate activation ranges after smoothing
    set_mode(m, "static")

    _, _, ppl = evaluate(m, val_loader, device)   # use VAL set for tuning
    alpha_results[alpha] = ppl
    print(f"alpha={alpha:.2f}  val PPL={ppl:.2f}")
    del m

best_alpha = min(alpha_results, key=alpha_results.get)
print(f"\nBest alpha: {best_alpha}  (val PPL={alpha_results[best_alpha]:.2f})")

In [ ]:
# ── Evaluate all 4 SmoothQuant configs with best alpha ───────────────────────
best_scales = compute_smooth_scales(orig, calib_loader, device, alpha=best_alpha)
results_22  = {}

for label, w_bw, mode in CONFIGS:
    m = copy.deepcopy(orig)
    if mode == "static":
        wrap_gpt2(m, w_bw=w_bw, a_bw=8, mode="observe", smooth=best_scales)
        m.to(device)
        calibrate(m, calib_loader, device)
        set_mode(m, "static")
    else:
        wrap_gpt2(m, w_bw=w_bw, a_bw=8, mode="dynamic", smooth=best_scales)
        m.to(device)

    acc, nll, ppl = evaluate(m, test_loader, device)
    w_err         = weight_recon_errors(baseline, m)
    a_err         = activation_recon_errors(m, val_loader, device)

    results_22[label] = dict(
        acc=acc, nll=nll, ppl=ppl,
        w_err_mean=np.mean(list(w_err.values())),
        a_err_mean=np.mean(list(a_err.values())),
        w_err=w_err, a_err=a_err,
    )
    print(f"SQ {label:15s}  Acc={acc:.2f}%  NLL={nll:.4f}  PPL={ppl:.2f}  "
          f"w_err={results_22[label]['w_err_mean']:.4f}  a_err={results_22[label]['a_err_mean']:.4f}")
    del m

In [ ]:
# ── Post-smoothing distribution (same blocks/projections) ────────────────────
# Use a smoothed+unwrapped model to collect activations after D^{-1} scaling

m_smooth_fp = copy.deepcopy(orig)
wrap_gpt2(m_smooth_fp, w_bw=None, a_bw=8, mode="off", smooth=best_scales)
m_smooth_fp.to(device)

act_data_smooth = {}
handles         = []

for i in INSPECT_BLOCKS:
    for parent, attr in INSPECT_PROJS:
        wrapper = getattr(getattr(m_smooth_fp.transformer.h[i], parent), attr)
        key     = f"h{i}.{parent}.{attr}"
        act_data_smooth[key] = []

        def make_hook(k, w):
            def hook(mod, inp):
                # input to QuantWrapper; smooth division happens inside forward
                x = inp[0].detach().cpu().float()
                if w.smooth is not None:
                    x = x / w.smooth.cpu().float()
                act_data_smooth[k].append(x.view(-1, x.size(-1)))
            return hook
        handles.append(wrapper.register_forward_pre_hook(make_hook(key, wrapper)))

with torch.no_grad():
    for (ids,) in calib_loader:
        m_smooth_fp(ids.to(device), use_cache=False)
for h in handles:
    h.remove()

act_data_smooth = {k: torch.cat(v, dim=0) for k, v in act_data_smooth.items()}

# Plot comparison
fig, axes = plt.subplots(len(INSPECT_BLOCKS), len(INSPECT_PROJS) * 2,
                         figsize=(16, 4 * len(INSPECT_BLOCKS)))

for row, i in enumerate(INSPECT_BLOCKS):
    for col, (parent, attr) in enumerate(INSPECT_PROJS):
        key = f"h{i}.{parent}.{attr}"
        ax  = axes[row, col * 2]
        ax.hist(act_data[key].numpy().ravel(),        bins=100, alpha=0.6, label="Before", color="steelblue")
        ax.hist(act_data_smooth[key].numpy().ravel(), bins=100, alpha=0.6, label="After",  color="darkorange")
        ax.set_title(f"Block {i} {parent}.{attr} – activation dist")
        ax.legend()

        ax2  = axes[row, col * 2 + 1]
        bef  = np.abs(act_data[key].numpy()).max(axis=0)
        aft  = np.abs(act_data_smooth[key].numpy()).max(axis=0)
        ax2.plot(np.sort(bef)[::-1], label="Before", color="steelblue")
        ax2.plot(np.sort(aft)[::-1], label="After",  color="darkorange")
        ax2.set_title(f"Block {i} {parent}.{attr} – per-channel max |x'|")
        ax2.legend()

        print(f"{key}  before max={bef.max():.3f}  after max={aft.max():.3f}  "
              f"before p99={np.percentile(bef,99):.3f}  after p99={np.percentile(aft,99):.3f}")

plt.tight_layout()
plt.savefig("dist_after_smooth.png", dpi=120)
plt.show()
del m_smooth_fp

In [ ]:
# Profile all 4 SmoothQuant configs
profile_22 = {}

for label, w_bw, mode in CONFIGS:
    m = copy.deepcopy(orig)
    if mode == "static":
        wrap_gpt2(m, w_bw=w_bw, a_bw=8, mode="observe", smooth=best_scales)
        m.to(device)
        calibrate(m, calib_loader, device)
        set_mode(m, "static")
    else:
        wrap_gpt2(m, w_bw=w_bw, a_bw=8, mode="dynamic", smooth=best_scales)
        m.to(device)

    r = gpt2p.profile(m, input_shape=(1, 128), device=device)
    profile_22[label] = r
    print(f"SQ {label:15s}  lat={r['total_latency']*1e3:.2f}ms  "
          f"mem={r['gpu_mem_peak']/1e6:.1f}MB  energy={r['energy_j']:.3f}mJ")
    del m

In [ ]:
# ── Consolidated results table (2.4) ────────────────────────────────────────
# FP32 baseline row uses Task 1 values; fill in after running Task 1.

print(f"{'Config':<22} {'Method':<10} {'Acc%':>6} {'NLL':>7} {'PPL':>7} {'Lat ms':>8} {'Tok/s':>8} {'W-err':>8} {'A-err':>8}")
print("-" * 90)

for label, _, _ in CONFIGS:
    r21 = results_21[label]
    pr  = profile_21[label]
    print(f"{label:<22} {'LinearPTQ':<10} {r21['acc']:>6.2f} {r21['nll']:>7.4f} {r21['ppl']:>7.2f} "
          f"{pr['total_latency']*1e3:>8.2f} {128/pr['total_latency']:>8.1f} "
          f"{r21['w_err_mean']:>8.4f} {r21['a_err_mean']:>8.4f}")

print()
for label, _, _ in CONFIGS:
    r22 = results_22[label]
    pr  = profile_22[label]
    print(f"{label:<22} {'SmoothQuant':<10} {r22['acc']:>6.2f} {r22['nll']:>7.4f} {r22['ppl']:>7.2f} "
          f"{pr['total_latency']*1e3:>8.2f} {128/pr['total_latency']:>8.1f} "
          f"{r22['w_err_mean']:>8.4f} {r22['a_err_mean']:>8.4f}")

---
## Task 3 – ResNet-18 Quantization-Aware Training on CIFAR-10

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.transforms import v2
from sklearn.metrics import f1_score

import resnet as rn
import profiler
from qat import replace_layers_qat, freeze_observers, qat_recon_errors, train_qat

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2471, 0.2435, 0.2616)

In [ ]:
# ── Data loaders for Task 3 ───────────────────────────────────────────────────
# Train: 45,000  |  Val: 5,000  (from CIFAR-10 train, seed=42)
# Test:  10,000  (official CIFAR-10 test split)

_aug_xform  = v2.Compose([
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip(),
    v2.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
_eval_xform = v2.Compose([
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

_g       = torch.Generator().manual_seed(42)
_all_idx = torch.randperm(50000, generator=_g).tolist()
_t_idx   = _all_idx[:45000]
_v_idx   = _all_idx[45000:]

from torch.utils.data import Subset as _Subset

t3_train_set = _Subset(CIFAR10("data", train=True,  transform=_aug_xform,  download=True), _t_idx)
t3_val_set   = _Subset(CIFAR10("data", train=True,  transform=_eval_xform, download=True), _v_idx)
t3_test_set  =         CIFAR10("data", train=False, transform=_eval_xform, download=True)

t3_train_loader = DataLoader(t3_train_set, batch_size=128, num_workers=2, shuffle=True,  pin_memory=True)
t3_val_loader   = DataLoader(t3_val_set,   batch_size=256, num_workers=2, shuffle=False, pin_memory=True)
t3_test_loader  = DataLoader(t3_test_set,  batch_size=256, num_workers=2, shuffle=False, pin_memory=True)

t3_dataloaders   = {"train": t3_train_loader, "val": t3_val_loader}
t3_dataset_sizes = {"train": len(t3_train_set), "val": len(t3_val_set)}

print(f"Train: {t3_dataset_sizes['train']}  Val: {t3_dataset_sizes['val']}  Test: {len(t3_test_set)}")

### 3.1 Pre-trained FP32 Baseline

Load the CIFAR-10–adapted ResNet-18 (3×3 stem, stride 1) pre-trained weights, evaluate on the test split, and profile on the GPU.

In [ ]:
# ── Load pretrained ResNet-18 + evaluation helper ─────────────────────────────
t3_fp32 = rn.resnet18(pretrained=True, device=device)
t3_fp32.eval().to(device)

@torch.no_grad()
def evaluate_clf(model, loader, device):
    """Returns (top1%, top5%, macro_f1%)."""
    model.eval()
    all_preds, all_top5, all_labels = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        top5   = logits.topk(5, dim=1).indices
        all_top5.append((top5 == labels.unsqueeze(1)).any(1).cpu())
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(labels.cpu())
    preds  = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    top1   = (preds == labels).float().mean().item() * 100
    top5   = torch.cat(all_top5).float().mean().item() * 100
    f1     = f1_score(labels.numpy(), preds.numpy(), average="macro") * 100
    return top1, top5, f1

t3_fp32_top1, t3_fp32_top5, t3_fp32_f1 = evaluate_clf(t3_fp32, t3_test_loader, device)
print(f"FP32  top-1={t3_fp32_top1:.2f}%  top-5={t3_fp32_top5:.2f}%  macro-F1={t3_fp32_f1:.2f}%")

In [ ]:
# ── FP32 profiling (batch=1, 32×32 image) ─────────────────────────────────────
t3_fp32_profile = profiler.profile(t3_fp32, input_shape=(1, 3, 32, 32), device=device)

n_params  = sum(p.numel() for p in t3_fp32.parameters())
lat_ms    = t3_fp32_profile["total_latency"] * 1e3
power_mw  = t3_fp32_profile["avg_power_w"]
energy_mj = t3_fp32_profile["energy_j"]

print(f"Parameters      : {n_params:,}")
print(f"MACs            : {t3_fp32_profile['macs']}")
print(f"FLOPs           : {t3_fp32_profile['flops']}")
print(f"Latency (ms)    : {lat_ms:.3f}")
print(f"GPU mem peak MB : {t3_fp32_profile['gpu_mem_peak'] / 1e6:.2f}")
print(f"Avg power (mW)  : {power_mw:.1f}")
print(f"Energy (mJ)     : {energy_mj:.4f}")

### 3.2 Quantization-Aware Training

**QAT** inserts *fake quantization* into the forward pass during training so the model learns to be robust to the rounding errors it will face at inference time.

**Fake quantization** with straight-through estimator (STE):

$$\hat{x} = \text{FakeQuant}(x) = (Q - z) \cdot s, \qquad Q = \operatorname{clip}\!\left(\operatorname{round}\!\left(\frac{x}{s}\right) + z,\ q_{\min},\ q_{\max}\right)$$

**STE backward pass** — the rounding step has zero gradient almost everywhere, so we pass the gradient straight through within the saturation range:

$$\frac{\partial \hat{x}}{\partial x} \approx \mathbf{1}\!\left[x \in \bigl[(q_{\min}-z)\cdot s,\ (q_{\max}-z)\cdot s\bigr]\right]$$

**Weight quantization** — symmetric per-output-channel INT$b$, zero-point $z = 0$:
$$s_c^W = \frac{\max_j |W_{cj}|}{Q_b}, \qquad Q_b = 2^{b-1} - 1$$

**Activation quantization** — asymmetric per-tensor unsigned INT8; scale and zero-point are estimated from running min/max statistics over training batches. Statistics are frozen at `freeze_epoch` to stabilise training.

Both `Conv2d` and the final `Linear` classifier are quantized; BatchNorm layers remain in FP32.

**Training recipe:** SGD, momentum 0.9, cosine LR schedule; 15 epochs with observer freeze at epoch 8 (W8A8) and 20 epochs with freeze at epoch 10 (W4A8).

In [ ]:
# ── W8A8 QAT ──────────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()

t3_w8a8 = copy.deepcopy(t3_fp32)
replace_layers_qat(t3_w8a8, bw_w=8, bw_a=8)

opt_w8a8 = torch.optim.SGD(t3_w8a8.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
sch_w8a8 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_w8a8, T_max=15)

t3_w8a8, hist_w8a8 = train_qat(
    t3_w8a8, criterion, t3_dataloaders, t3_dataset_sizes,
    opt_w8a8, sch_w8a8, device, epochs=15, freeze_epoch=8,
)
t3_w8a8.eval()

In [ ]:
# ── W4A8 QAT ──────────────────────────────────────────────────────────────────
t3_w4a8 = copy.deepcopy(t3_fp32)
replace_layers_qat(t3_w4a8, bw_w=4, bw_a=8)

opt_w4a8 = torch.optim.SGD(t3_w4a8.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
sch_w4a8 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_w4a8, T_max=20)

t3_w4a8, hist_w4a8 = train_qat(
    t3_w4a8, criterion, t3_dataloaders, t3_dataset_sizes,
    opt_w4a8, sch_w4a8, device, epochs=20, freeze_epoch=10,
)
t3_w4a8.eval()

### 3.3 Results and Profiling

In [ ]:
# ── Test-set evaluation ────────────────────────────────────────────────────────
t3_results = {}
for label, model in [("FP32", t3_fp32), ("QAT W8A8", t3_w8a8), ("QAT W4A8", t3_w4a8)]:
    top1, top5, f1 = evaluate_clf(model, t3_test_loader, device)
    t3_results[label] = dict(top1=top1, top5=top5, f1=f1)
    print(f"{label:<12}  top-1={top1:.2f}%  top-5={top5:.2f}%  macro-F1={f1:.2f}%")

In [ ]:
# ── Layer-wise weight reconstruction errors ────────────────────────────────────
for label, model in [("QAT W8A8", t3_w8a8), ("QAT W4A8", t3_w4a8)]:
    errs     = qat_recon_errors(model)
    mean_err = np.mean(list(errs.values()))
    print(f"\n{label}  mean weight recon error: {mean_err:.6f}")
    print(f"  {'Layer':<38}  {'Recon Error':>12}")
    print("  " + "-" * 52)
    for name, e in sorted(errs.items(), key=lambda x: -x[1])[:10]:
        print(f"  {name:<38}  {e:>12.6f}")

In [ ]:
# ── Profiling (batch=1, 32×32 image) ─────────────────────────────────────────
t3_profiles = {}
for label, model in [("FP32", t3_fp32), ("QAT W8A8", t3_w8a8), ("QAT W4A8", t3_w4a8)]:
    r = profiler.profile(model, input_shape=(1, 3, 32, 32), device=device)
    t3_profiles[label] = r
    print(
        f"{label:<12}  lat={r['total_latency']*1e3:.3f}ms  "
        f"mem={r['gpu_mem_peak']/1e6:.2f}MB  "
        f"power={r['avg_power_w']:.1f}mW  energy={r['energy_j']:.4f}mJ"
    )

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, hist, ls in [("W8A8", hist_w8a8, "-"), ("W4A8", hist_w4a8, "--")]:
    epochs = range(1, len(hist["train_acc"]) + 1)
    axes[0].plot(epochs, [a * 100 for a in hist["train_acc"]], ls,     label=f"{label} train")
    axes[0].plot(epochs, [a * 100 for a in hist["val_acc"]],   ls,     label=f"{label} val",   alpha=0.7)
    axes[1].plot(epochs, hist["train_loss"],                   ls,     label=f"{label} train")
    axes[1].plot(epochs, hist["val_loss"],                     ls,     label=f"{label} val",   alpha=0.7)

axes[0].set_title("Accuracy over QAT epochs")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy (%)")
axes[0].legend()
axes[1].set_title("Loss over QAT epochs")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Cross-Entropy Loss")
axes[1].legend()
plt.tight_layout()
plt.savefig("qat_training_curves.png", dpi=120)
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
import tempfile as _tmp, os as _os

def _serialized_mb(m):
    with _tmp.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        p = f.name
    torch.save(m.state_dict(), p)
    mb = _os.path.getsize(p) / 1e6
    _os.remove(p)
    return mb

def _theory_mb(model, bits_w):
    n_q, n_fp = 0, 0
    from qat import QATConv2d, QATLinear
    for m in model.modules():
        if isinstance(m, (QATConv2d, QATLinear)):
            n_q += m.weight.numel()
        elif isinstance(m, (nn.Conv2d, nn.Linear)):
            n_fp += m.weight.numel()
    return (n_q * bits_w / 8 + n_fp * 4) / 1e6

hdr = f"{'Config':<12} {'Top-1%':>7} {'Top-5%':>7} {'F1%':>7} {'Size MB':>8} {'Theory MB':>10} {'Lat ms':>8} {'Mem MB':>8} {'Energy mJ':>10}"
print(hdr)
print("-" * len(hdr))

bits = {"FP32": 32, "QAT W8A8": 8, "QAT W4A8": 4}
for label, model in [("FP32", t3_fp32), ("QAT W8A8", t3_w8a8), ("QAT W4A8", t3_w4a8)]:
    r   = t3_results[label]
    pr  = t3_profiles[label]
    smb = _serialized_mb(model)
    tmb = _theory_mb(model, bits[label])
    print(
        f"{label:<12} {r['top1']:>7.2f} {r['top5']:>7.2f} {r['f1']:>7.2f} "
        f"{smb:>8.2f} {tmb:>10.2f} "
        f"{pr['total_latency']*1e3:>8.3f} {pr['gpu_mem_peak']/1e6:>8.2f} "
        f"{pr['energy_j']:>10.4f}"
    )

### 3.4 Analysis

*(Fill in after running the cells above.)*

**Accuracy–compression trade-off:**
- FP32 baseline: record actual top-1/top-5/F1 from the cells above.
- W8A8 QAT: compare top-1/top-5/F1 vs FP32; expect less than 0.5 pp drop at this bit-width.
- W4A8 QAT: report additional accuracy drop vs W8A8; 4-bit weight quantization typically costs 1–3 pp on CIFAR-10.

**Effect of observer freezing:**
- Training accuracy typically stabilises after the freeze epoch because the model stops adapting to shifting activation statistics.
- Instability before the freeze epoch reflects activation ranges that are still warming up.

**Reconstruction errors:**
- W8A8 per-layer weight reconstruction errors should be negligible (< 1 × 10⁻⁴).
- W4A8 errors are expected to be roughly 16× higher due to the smaller quantization grid.
- Layers with large weight dynamic ranges (e.g., the final `fc` layer) usually have higher relative errors.

**Memory and latency:**
- The fake-quantization training path keeps weights in FP32, so the serialised model size equals FP32. Packed INT8/INT4 sizes are reported separately as the theoretical deployment footprint.
- GPU latency may not decrease relative to FP32 because PyTorch fake-quantization does not dispatch INT8 CUDA kernels; actual speedup requires an INT8 inference runtime (e.g., TensorRT, ONNX-RT).